# Train models

Trains one model per target (receptions, receiving yards, receiving TDs) with the features chosen in `config.py`, then saves the models and metrics to `models/`. Run once per offseason.

In [ ]:
import sys
sys.path.append('..')

from src import config, features

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_poisson_deviance, mean_squared_error, r2_score
from xgboost import XGBRegressor, plot_importance

## 1. Load the training data
Built by `01_build_training_data`.

In [ ]:
table = pd.read_parquet(config.TRAINING_DATA)
print(table.shape, 'seasons', table['season'].min(), '-', table['season'].max())

## 2. Compare the training row methods
Three seasons, three roles:
- **Train** (every season before `config.VALIDATION_SEASON`): the model learns from these games.
- **Validation** (`config.VALIDATION_SEASON`): picks the number of trees with early stopping. Trees are added one at a time and training stops once the validation score hasn't improved for `config.EARLY_STOPPING_ROUNDS` trees, before the model starts memorising noise. Targets with a fixed `n_estimators` in `config.XGB_PARAMS` skip this.
- **Test** (`config.TEST_SEASON`): the model is refit on every earlier season with that number of trees and scored once on this season.

Other choices:
- **Same validation and test rows for both methods**: WR games picked with the snap rule (previous game >= 40% snaps). It only uses information known before the game, like the weekly predictions, while the yards rule needs the season's final total.
- **Held-out seasons instead of a random split**: with a random split, the averages of a test game overlap with the games around it in training, which makes the scores look better than they are.
- **Reference rows**: the training average for every player, and the player's last 3 games average (career average if missing), to see how much the model adds.

Lower RMSE and higher R2 are better. **For TDs, use RMSE and Poisson deviance, not MAE**: MAE rewards predicting the median, which is 0, so predicting 0 TDs for everyone would get the best MAE.

In [ ]:
def tree_count(rows, target):
    """Number of trees: fixed in config, or picked by early stopping on the validation season."""
    params = config.XGB_PARAMS[target]
    if 'n_estimators' in params:
        return params['n_estimators']
    feats = config.MODEL_FEATURES[target]
    fit_rows = rows[rows['season'] < config.VALIDATION_SEASON]
    model = XGBRegressor(**params, n_estimators=config.MAX_TREES, eval_metric='rmse',
                         early_stopping_rounds=config.EARLY_STOPPING_ROUNDS)
    model.fit(fit_rows[feats], fit_rows[target], eval_set=[(validation[feats], validation[target])], verbose=False)
    return model.best_iteration + 1


def train(rows, target, n_trees):
    model = XGBRegressor(**{**config.XGB_PARAMS[target], 'n_estimators': n_trees})
    model.fit(rows[config.MODEL_FEATURES[target]], rows[target])
    return model


def scores(y, pred):
    result = {'RMSE': mean_squared_error(y, pred) ** 0.5,
              'MAE': mean_absolute_error(y, pred),
              'R2': r2_score(y, pred)}
    if (y >= 0).all():  # Poisson deviance needs non-negative values (yards can be negative)
        result['Poisson dev'] = mean_poisson_deviance(y, np.clip(pred, 1e-6, None))
    result['mean pred'] = np.mean(pred)
    return result


before_test = table[table['season'] < config.TEST_SEASON]
validation = features.select_training_rows(table[table['season'] == config.VALIDATION_SEASON], 'snap')
test = features.select_training_rows(table[table['season'] == config.TEST_SEASON], 'snap')
print(f'Validation rows ({config.VALIDATION_SEASON}): {len(validation):,}   Test rows ({config.TEST_SEASON}): {len(test):,}')

results = []
for target in config.TARGETS:
    y = test[target]
    snap_rows = features.select_training_rows(before_test, 'snap')
    average = np.full(len(y), snap_rows[target].mean())
    baseline = test[f'{target}_last{config.N_GAMES}'].fillna(test[f'{target}_career']).fillna(0)
    results.append({'target': target, 'method': 'training average for everyone', **scores(y, average)})
    results.append({'target': target, 'method': f'baseline (last {config.N_GAMES} avg)', **scores(y, baseline)})
    for method in ['snap', 'yards']:
        rows = features.select_training_rows(before_test, method)
        n_trees = tree_count(rows, target)
        model = train(rows, target, n_trees)
        pred = model.predict(test[config.MODEL_FEATURES[target]])
        results.append({'target': target, 'method': method, 'train rows': len(rows), 'trees': n_trees, **scores(y, pred)})

comparison = pd.DataFrame(results).set_index(['target', 'method']).round(3)
comparison

## 3. Train and save the final models
Trains on every season in `config.TRAIN_YEARS` (validation and test seasons included) with the rows from `config.TRAINING_METHOD`, using the number of trees picked the same way as above. Saves one model per target to `models/` with the comparison above.

In [ ]:
config.MODELS_DIR.mkdir(exist_ok=True)
rows = features.select_training_rows(table, config.TRAINING_METHOD)
print(f'Training rows ({config.TRAINING_METHOD}): {len(rows):,}')

models = {}
for target in config.TARGETS:
    n_trees = tree_count(rows, target)
    models[target] = train(rows, target, n_trees)
    models[target].save_model(config.model_path(target))
    print(f'Saved {config.model_path(target).name} ({n_trees} trees)')

metrics = comparison.reset_index()
metrics['final_method'] = config.TRAINING_METHOD
metrics.to_csv(config.MODEL_METRICS, index=False)

## 4. Feature importance

In [ ]:
fig, axes = plt.subplots(1, len(config.TARGETS), figsize=(20, 7))
for ax, target in zip(axes, config.TARGETS):
    plot_importance(models[target], ax=ax, max_num_features=15, importance_type='gain',
                    title=target, show_values=False)
plt.tight_layout()
plt.show()